# 08 - Cohere com LangChain

O `ChatCohere` integra os modelos Cohere ao ecossistema LangChain.
Mesma interface dos outros provedores — só muda o import.

### Pacotes necessários

```bash
pip install langchain-cohere python-dotenv
```

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_cohere import ChatCohere

load_dotenv(find_dotenv())

chat = ChatCohere(model="command-r-plus", temperature=1)

response = chat.invoke("E aí, tudo bem?")
print(response.content)

## 1. Chain com múltiplas variáveis no prompt

O `ChatCohere` funciona com qualquer prompt template do LangChain.
Aqui usamos dois parâmetros — `{conteudo}` e `{n_palavras}` — para controlar a resposta.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = ChatPromptTemplate.from_messages([
    ("system", "Você é um contador de histórias"),
    ("human", "Conte uma história sobre {conteudo} com até {n_palavras} palavras")
])

chain = template | chat | StrOutputParser()

# História curta
response = chain.invoke({"conteudo": "análise de dados", "n_palavras": 20})
print("História curta:")
print(response)
print()

# História média
response2 = chain.invoke({"conteudo": "data science", "n_palavras": 30})
print("História média:")
print(response2)

## 2. Streaming com LCEL

In [ ]:
for chunk in chain.stream({"conteudo": "inteligência artificial", "n_palavras": 200}):
    print(chunk, end="", flush=True)

## Resumo — Comparativo de todos os provedores

Agora que vimos todos os provedores, aqui está o comparativo completo:

| Provedor | LangChain class | Pacote | Variável `.env` |
|----------|----------------|--------|----------------|
| OpenAI | `ChatOpenAI` | `langchain-openai` | `OPENAI_API_KEY` |
| Groq | `ChatGroq` | `langchain-groq` | `GROQ_API_KEY` |
| Google Gemini | `ChatGoogleGenerativeAI` | `langchain-google-genai` | `GEMINI_API_KEY` |
| Mistral | `ChatMistralAI` | `langchain-mistralai` | `MISTRAL_API_KEY` |
| Cohere | `ChatCohere` | `langchain-cohere` | `COHERE_API_KEY` |

**A beleza do LangChain**: a chain abaixo funciona com qualquer provedor apenas trocando a linha do modelo:

```python
# Troque apenas esta linha para mudar de provedor
chat = ChatOpenAI(model="gpt-4o-mini")          # OpenAI
chat = ChatGroq(model="llama-3.3-70b-versatile") # Groq
chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash") # Gemini
chat = ChatMistralAI(model="mistral-large-latest")       # Mistral
chat = ChatCohere(model="command-r-plus")         # Cohere

# O resto da chain permanece idêntico
chain = template | chat | StrOutputParser()
response = chain.invoke({"conteudo": "IA", "n_palavras": 50})
```